<a href="https://colab.research.google.com/github/syedmahmoodiagents/transformers/blob/main/Full_transformer_translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import spacy
import torch
import torch.nn as nn

torch.manual_seed(42)

nlp = spacy.load("en_core_web_sm")

english_corpus = [
    "I love mathematics",
    "calculus is great",
    "mathematics is useful",
    "I love calculus",
    "mathematics is great"
]

dutch_corpus = [
    "ik hou van wiskunde",
    "calculus is geweldig",
    "wiskunde is nuttig",
    "ik hou van calculus",
    "wiskunde is geweldig"
]

SPECIAL = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]

def build_vocab(sentences):
    w2i={t:i for i,t in enumerate(SPECIAL)}
    i2w={i:t for i,t in enumerate(SPECIAL)}

    idx=len(SPECIAL)
    for s in sentences:
        for tok in nlp(s):
            w=tok.text.lower()
            if w not in w2i:
                w2i[w]=idx
                i2w[idx]=w
                idx+=1
    return w2i,i2w

src_w2i,src_i2w=build_vocab(english_corpus)
tgt_w2i,tgt_i2w=build_vocab(dutch_corpus)

def encode_src(s):
    return [src_w2i.get(t.text.lower(), src_w2i["<UNK>"]) for t in nlp(s)]

def encode_tgt(s):
    ids=[tgt_w2i["<SOS>"]]
    ids.extend([tgt_w2i.get(t.text.lower(),tgt_w2i["<UNK>"]) for t in nlp(s)])
    ids.append(tgt_w2i["<EOS>"])
    return ids

src=[encode_src(s) for s in english_corpus]
tgt=[encode_tgt(s) for s in dutch_corpus]

src_len=max(map(len,src))
tgt_len=max(map(len,tgt))


In [ ]:
def pad(seq,l,pad=0):
    return seq+[pad]*(l-len(seq))

src=torch.tensor([pad(x,src_len) for x in src])
tgt=torch.tensor([pad(x,tgt_len) for x in tgt])

decoder_input=tgt[:,:-1] # one word short of last word before <EOS>
target=tgt[:,1:] # decoder output counting after <SOS>

In [ ]:


class PositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=100):
        super().__init__()
        pe=torch.zeros(max_len,d_model)
        pos=torch.arange(max_len).unsqueeze(1)

        div=torch.exp(torch.arange(0,d_model,2)*(-math.log(10000)/d_model))
        pe[:,0::2]=torch.sin(pos*div)
        pe[:,1::2]=torch.cos(pos*div)

        self.pe = pe.unsqueeze(0)

    def forward(self,x):
        return x+self.pe[:,:x.size(1)]

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,heads):
        super().__init__()
        self.h=heads
        self.d=d_model//heads

        self.q=nn.Linear(d_model,d_model)
        self.k=nn.Linear(d_model,d_model)
        self.v=nn.Linear(d_model,d_model)
        self.out=nn.Linear(d_model,d_model)

    def forward(self,q,k,v,mask=None):
        B=q.size(0)
        Q=self.q(q).view(B,-1,self.h,self.d).transpose(1,2)
        K=self.k(k).view(B,-1,self.h,self.d).transpose(1,2)
        V=self.v(v).view(B,-1,self.h,self.d).transpose(1,2)
        scores=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(self.d)

        if mask is not None:
            scores=scores.masked_fill(mask==0,-1e9)

        attn=torch.softmax(scores,-1)
        ctx=torch.matmul(attn,V)
        ctx=ctx.transpose(1,2).contiguous().view(B, -1, self.h*self.d)
        return self.out(ctx)

In [ ]:


class FeedForward(nn.Module):
    def __init__(self,d_model):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(d_model,4*d_model),
            nn.ReLU(),
            nn.Linear(4*d_model,d_model)
        )
    def forward(self,x): return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self,d_model,heads):
        super().__init__()
        self.attn=MultiHeadAttention(d_model,heads)
        self.n1=nn.LayerNorm(d_model)
        self.ff=FeedForward(d_model)
        self.n2=nn.LayerNorm(d_model)

    def forward(self,x):
        x=self.n1(x+self.attn(x,x,x))
        x=self.n2(x+self.ff(x))
        return x

class DecoderLayer(nn.Module):
    def __init__(self,d_model,heads):
        super().__init__()
        self.self_attn=MultiHeadAttention(d_model,heads)
        self.n1=nn.LayerNorm(d_model)

        self.cross=MultiHeadAttention(d_model,heads)

        self.n2=nn.LayerNorm(d_model)
        self.ff=FeedForward(d_model)
        self.n3=nn.LayerNorm(d_model)

    def forward(self,x,enc,mask):
        x=self.n1(x+self.self_attn(x,x,x,mask))
        x=self.n2(x+self.cross(x,enc,enc))
        x=self.n3(x+self.ff(x))
        return x


In [ ]:

class Transformer(nn.Module):
    def __init__(self,src_vocab,tgt_vocab,d_model=64,heads=4):
        super().__init__()
        self.src_emb=nn.Embedding(src_vocab,d_model)
        self.tgt_emb=nn.Embedding(tgt_vocab,d_model)

        self.pos=PositionalEncoding(d_model)
        self.encoder=EncoderLayer(d_model,heads)
        self.decoder=DecoderLayer(d_model,heads)
        self.fc=nn.Linear(d_model,tgt_vocab)

    def causal_mask(self,n):
        m=torch.tril(torch.ones(n,n))
        return m.unsqueeze(0).unsqueeze(0)

    def forward(self,src,tgt):
        src=self.pos(self.src_emb(src))
        enc=self.encoder(src)
        tgt=self.pos(self.tgt_emb(tgt))
        mask=self.causal_mask(tgt.size(1))
        dec=self.decoder(tgt,enc,mask)
        return self.fc(dec)


In [ ]:
model=Transformer(len(src_w2i) ,len(tgt_w2i))

loss_fn=nn.CrossEntropyLoss(ignore_index=tgt_w2i["<PAD>"])
opt=torch.optim.Adam(model.parameters(),lr=0.005)

In [ ]:
for epoch in range(500):
    opt.zero_grad()
    out=model(src,decoder_input)
    loss=loss_fn(out.reshape(-1, len(tgt_w2i)), target.reshape(-1))
    loss.backward()
    opt.step()
    if (epoch+1)%50==0:
        print(epoch+1,loss.item())

50 0.0019568309653550386
100 0.0009467501658946276
150 0.0006229541613720357
200 0.0004444046935532242
250 0.00033421433181501925
300 0.0002611622039694339
350 0.0002101208665408194
400 0.00017290805408265442
450 0.00014497987285722047
500 0.00012329229502938688


In [ ]:
def translate(sentence):
    model.eval()
    s=torch.tensor([pad(encode_src(sentence),src_len)])
    generated=[tgt_w2i["<SOS>"]]
    with torch.no_grad():
        for _ in range(tgt_len):
            dec=torch.tensor([generated])
            out=model(s,dec)
            nxt=out[0,-1].argmax().item()
            if nxt==tgt_w2i["<EOS>"]:
                break
            generated.append(nxt)
    words=[tgt_i2w[i] for i in generated[1:]]
    return " ".join(words)


In [ ]:

for s in english_corpus:
    print(s,"->",translate(s))

I love mathematics -> ik hou van wiskunde
calculus is great -> calculus is geweldig
mathematics is useful -> wiskunde is nuttig
I love calculus -> ik hou van calculus
mathematics is great -> wiskunde is geweldig
